### Running cell2location to map our 10X Genomics Bleo day21 data to publicly available 10X Visium Bleo day21 data

ST data from [Franzen, Lindvall, et al. Nature Genetics (2024)](https://www.nature.com/articles/s41588-024-01819-2)

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sb

In [2]:
import cell2location as c2l

/home/niklas/miniconda3/envs/c2l_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# making sure plots & clusters are reproducible
np.random.seed(42)

In [4]:
## plotting variables
sc.settings.figdir = '/home/niklas/projects/GLPG_drug_perturbations/02_figures/fig_spatial_overview/'
sc.set_figure_params(vector_friendly = True)
plt.rcParams['figure.figsize'] = (6, 5)
plt.rcParams['pdf.fonttype'] = 42

In [5]:
## path variables
sc_dir = '/mnt/smb/Niklas/20240911_10XVisum_Stockholm_mouse/reference_signatures/240923_GLPG_PBS_Bleo_d21_reference_signatures.h5ad'
spatial_dir = '/mnt/smb/Niklas/20240911_10XVisum_Stockholm_mouse/240919_10XVisium_d21.h5ad'
data_dir = '/mnt/smb/Niklas/20240911_10XVisum_Stockholm_mouse/'

In [6]:
## paths reference regression and cell2location models
ref_run_name = f'{data_dir}/reference_signatures'
run_name = f'{data_dir}/cell2location_map'

### Load Visium data

In [7]:
adata_st = sc.read(spatial_dir)

### Load scRNA-seq reference data

In [8]:
adata_sc = sc.read(sc_dir)

In [9]:
adata_sc

AnnData object with n_obs × n_vars = 72828 × 11916
    obs: 'identifier', 'time_point', 'n_counts', 'preprocessing_cluster', 'size_factors', 'S_score', 'G2M_score', 'phase', 'treatment_time', 'treatment', 'sample_name', 'condition', 'percent_mito', 'cluster_03', 'doublet_scores', 'n_genes', 'louvain_1', 'louvain_2', 'cell_type', 'group', 'ChromiumBatch', 'LibraryPrepBatch', 'SeqBatch', 'ashcroft_score', 'body_weight_d0', 'body_weight_d1', 'body_weight_d2', 'body_weight_d3', 'body_weight_d4', 'body_weight_d5', 'body_weight_d6', 'body_weight_d7', 'body_weight_d8', 'body_weight_d9', 'body_weight_d10', 'body_weight_d11', 'body_weight_d12', 'body_weight_d13', 'body_weight_d14', 'body_weight_d15', 'body_weight_d16', 'body_weight_d17', 'body_weight_d18', 'body_weight_d19', 'body_weight_d20', 'body_weight_d21', 'body_weight_min', 'body_weight_max', 'delta_max_body_weight', 'delta_body_weight_d7_10', 'delta_body_weight_d14_d21', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'n_counts', 'n_

### Load model

In [10]:
## default, try on GPU:
#use_gpu = False
#model = c2l.models.RegressionModel.load(f'{ref_run_name}', adata_sc)

### Extract estimated gene expression per cell type

In [11]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_sc.varm.keys():
    inf_aver = adata_sc.varm['means_per_cluster_mu_fg'][
        [f'means_per_cluster_mu_fg_{i}' for i in adata_sc.uns['mod']['factor_names']]
    ].copy()
else:
    inf_aver = adata_sc.var[
        [f'means_per_cluster_mu_fg_{i}' for i in adata_sc.uns['mod']['factor_names']]
    ].copy()

inf_aver.columns = adata_sc.uns['mod']['factor_names']
inf_aver.head()

,AT1,AT2,AT2 activated,Krt8+ ADI,Basal,Club/Ciliated,Ciliated,NEC,Aerocyte capillary EC,Transitional capillary EC,...,Cd4+ T cells,Cd8+ T cells,Cd4+/Cd8+ T cells,T reg cells,Th2 cells,Th17 cells,Themis+ T cells,Proliferating T cells,NK T cells,NK cells
ENSEMBL_ID,,,,,,,,,,,,,,,,,,,,,
ENSMUSG00000025902,0.004162,0.000579,0.003271,0.003578,0.014045,0.016038,0.001274,0.204295,0.224229,1.201024,...,0.000148,0.000236,0.001043,0.000504,0.001814,0.000481,0.007236,0.003447,0.000227,0.000286
ENSMUSG00000104238,0.005032,0.000576,0.004588,0.005302,0.023015,0.020713,0.001352,0.199595,0.006633,0.029319,...,0.000182,0.000294,0.000752,0.000754,0.001773,0.000739,0.007663,0.004958,0.000286,0.000299
ENSMUSG00000033845,0.062801,0.025044,0.089999,0.249837,0.209690,0.028407,0.176307,0.369460,0.117702,0.320646,...,0.091802,0.123577,0.059050,0.094917,0.117172,0.143549,0.146277,0.509909,0.125265,0.072292
ENSMUSG00000025903,0.099898,0.136380,0.083479,0.059058,0.224783,0.047520,0.191615,0.222509,0.162986,0.263908,...,0.043983,0.036538,0.025005,0.047501,0.045922,0.066150,0.036739,0.224329,0.065302,0.035774
ENSMUSG00000033813,0.129857,0.109014,0.109680,0.243608,0.212516,0.040569,0.257397,0.330919,0.128584,0.421502,...,0.068450,0.072395,0.058209,0.091557,0.134052,0.145132,0.080561,0.168519,0.112175,0.059647


In [12]:
## 
inf_aver.to_csv(ref_run_name + 'inf_aver.csv')

### Cell type mapping

In [13]:
## find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_st.var_names, inf_aver.index)
adata_st = adata_st[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

In [14]:
## prepare anndata
c2l.models.Cell2location.setup_anndata(
    adata=adata_st,
    batch_key='sample',
)

In [15]:
## create model
model = c2l.models.Cell2location(
    adata_st,
    cell_state_df = inf_aver,
    N_cells_per_location = 7,
    detection_alpha = 20
)
model.view_anndata_setup()

Anndata setup with scvi-tools version 1.2.2.post2.

Setup via `Cell2location.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'sample',
│   'labels_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃     Summary Stat Key     ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│         n_batch          │   6   │
│         n_cells          │ 23107 │
│ n_extra_categorical_covs │   0   │
│ n_extra_continuous_covs  │   0   │
│         n_labels         │   1   │
│          n_vars          │ 11916 │
└──────────────────────────┴───────┘

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

                    batch State Registry                     
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃   Source Location   ┃  Categories   ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['sample'] │ V10A20-071-A1 │          0          │
│                     │ V10A20-052-A1 │          1          │
│                     │ V10A20-053-A1 │          2          │
│                     │ V10A20-053-B1 │          3          │
│                     │ V10A20-053-C1 │          4          │
│                     │ V10A20-053-D1 │          5          │
└─────────────────────┴───────────────┴─────────────────────┘

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

In [16]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [ ]:
## train model
model.train(max_epochs=30000, batch_size=None, train_size=1, accelerator = 'cpu')
# plot training history
model.plot_history()

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [ ]:
adata_st = model.export_posterior(
    adata_st,
    sample_kwargs={
        'num_samples': 1000,
        'batch_size': model.adata.n_obs,
        'use_gpu': False,
    },
)

In [ ]:
model.plot_QC()

In [ ]:
adata_st.obs[adata_st.uns['mod']['factor_names']] = adata_st.obsm[
    'q05_cell_abundance_w_sf'
]

In [ ]:
## save model
model.save(f"{run_name}", overwrite=True)

In [ ]:
## save anndata object with results
adata_file = f"{run_name}/250111_10XVisium_d21_deconvolution.h5ad"
adata_st.write(adata_file)
adata_file

In [ ]:
#def select_slide(adata, s, s_col='sample'):
#    r""" This function selects the data for one slide from the spatial anndata object.
#
#    :param adata: Anndata object with multiple spatial experiments
#    :param s: name of selected experiment
#    :param s_col: column in adata.obs listing experiment name for each location
#    """
#
#    slide = adata[adata.obs[s_col].isin([s]), :]
#    s_keys = list(slide.uns['spatial'].keys())
#    s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]
#
#    slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}
#
#    return slide

In [ ]:
## select one slide for visualization
#for sample_name in adata_st.obs['sample'].unique():
#    slide = select_slide(adata_st, sample_name)
#
#    with mpl.rc_context({"figure.figsize": [4.5, 5]}):
#        sc.pl.spatial(
#            slide,
#            cmap="magma",
#            color=adata_st.uns["mod"]["factor_names"],
#            ncols=4,
#            size=1.3,
#            img_key="hires",
#            # limit color scale at 99.2% quantile of cell abundance
#            vmin=0,
#            vmax="p99.2",
#        )

In [ ]:
#clust_col = ['Eosinophils','Cthrc1= Myofibroblasts','Krt8+ ADI','Vwa1+/Col15a1+ ectopic EC']
#clust_labels = clust_col
#
## select one slide for visualization
#for sample_name in adata_st.obs['sample'].unique():
#    slide = select_slide(adata_st, sample_name)
#    with matplotlib.rc_context({"figure.figsize": (15, 15)}):
#    fig = c2l.plt.plot_spatial(
#        adata=slide,
#        color=clust_col,
#        labels=clust_labels,
#        max_color_quantile=0.992,
#        circle_diameter=6,
#        show_img=True,
#        colorbar_position="right",
#        colorbar_shape={"horizontal_gaps": 0.2},
#    )